In [ ]:
import pickle
import torch

In [ ]:
file = open('../data/CV data/train1.pkl', 'rb')
train_data = pickle.load(file)

In [ ]:
file = open('../data/CV data/val1.pkl', 'rb')
val_data = pickle.load(file)

In [1]:
import pickle
import torch

# 1. 加载数据
file_path_train = '../data/CV data/train1.pkl'
file_path_val = '../data/CV data/val1.pkl'

with open(file_path_train, 'rb') as f:
    train_data = pickle.load(f)

with open(file_path_val, 'rb') as f:
    val_data = pickle.load(f)

# --- 验证逻辑开始 ---

def inspect_structure(data, name):
    print(f"\n{'='*30} {name} 数据结构分析 {'='*30}")
    print(f"1. 数据类型: {type(data)}")
    
    # 情况 A: 如果是 PyTorch Geometric 的 Data 对象
    if hasattr(data, 'keys') and hasattr(data, 'x'): # 简单的启发式判断
        print("   -> 疑似 PyTorch Geometric Data 对象")
        print(f"   - 包含的键/属性: {data.keys()}")
        if hasattr(data, 'x'):
            print(f"   - 节点特征 (x) 形状: {data.x.shape}")
        if hasattr(data, 'edge_index'):
            print(f"   - 边索引 (edge_index) 形状: {data.edge_index.shape}")
        if hasattr(data, 'y'):
            print(f"   - 标签 (y) 形状: {data.y.shape}")
            
    # 情况 B: 如果是 Python 字典
    elif isinstance(data, dict):
        print("   -> Python 字典 (Dict)")
        print(f"   - 包含的键: {data.keys()}")
        for key, value in data.items():
            if hasattr(value, 'shape'):
                print(f"   - 键 '{key}' 的形状: {value.shape}")
            else:
                print(f"   - 键 '{key}' 的类型: {type(value)}")

    # 情况 C: 如果是列表
    elif isinstance(data, list):
        print(f"   -> Python 列表 (List)，长度: {len(data)}")
        if len(data) > 0:
            print(f"   - 第一个元素的类型: {type(data[0])}")
            # 如果列表里是图数据，尝试打印第一个图的形状
            if hasattr(data[0], 'x'):
                 print(f"   - 第一个图的节点特征形状: {data[0].x.shape}")
    
    else:
        print(f"   -> 其他类型: {type(data)}")

# 执行检查
inspect_structure(train_data, "训练集 (Train)")
inspect_structure(val_data, "验证集 (Val)")

c:\Users\86152\.conda\envs\pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



============================== 训练集 (Train) 数据结构分析 ==============================
1. 数据类型: <class 'torch_geometric.data.hetero_data.HeteroData'>
   -> 其他类型: <class 'torch_geometric.data.hetero_data.HeteroData'>

============================== 验证集 (Val) 数据结构分析 ==============================
1. 数据类型: <class 'torch_geometric.data.hetero_data.HeteroData'>
   -> 其他类型: <class 'torch_geometric.data.hetero_data.HeteroData'>


In [2]:
import torch

def inspect_hetero_data(data, name):
    print(f"\n{'='*40} {name} 数据详情 {'='*40}")
    
    # 1. 打印骨架：节点类型和边类型
    print(f"🔹 包含的节点类型: {data.node_types}")
    print(f"🔹 包含的边类型: {data.edge_types}")
    
    # 2. 遍历每种节点类型，查看具体数据
    for node_type in data.node_types:
        print(f"\n--- 节点类型: '{node_type}' ---")
        node_data = data[node_type]
        
        # 查看该类型节点有哪些属性 (如 x, y, train_mask 等)
        print(f"   属性键: {list(node_data.keys())}")
        
        # 查看特征矩阵 x
        if hasattr(node_data, 'x'):
            x = node_data.x
            print(f"   特征 (x) 形状: {x.shape} -> (节点数, 特征维度)")
            print(f"   特征数据类型: {x.dtype}")
            # 打印前 2 个节点的特征数据 (如果数据量够大)
            if x.shape[0] > 0:
                print(f"   前 2 个节点的特征值:\n{x[:2]}")
        
        # 查看标签 y
        if hasattr(node_data, 'y'):
            y = node_data.y
            print(f"   标签 (y) 形状: {y.shape}")
            print(f"   前 5 个标签值: {y[:5]}")
            
        # 查看掩码 (train_mask, val_mask 等)
        for key in node_data.keys():
            if 'mask' in key:
                mask = node_data[key]
                print(f"   掩码 ({key}) 形状: {mask.shape}, 训练/验证节点数: {mask.sum().item()}")

    # 3. 遍历每种边类型，查看连接关系
    for edge_type in data.edge_types:
        print(f"\n--- 边类型: {edge_type} ---")
        # edge_type 通常是一个三元组 ('源节点', '关系', '目标节点')
        src_type, relation, dst_type = edge_type
        edge_index = data[edge_type].edge_index
        
        print(f"   连接方式: [{src_type}] --({relation})--> [{dst_type}]")
        print(f"   边索引形状: {edge_index.shape} -> (2, 边数)")
        
        # 打印前 3 条边的连接关系
        if edge_index.shape[1] > 0:
            print(f"   前 3 条边 (源节点索引 -> 目标节点索引):")
            for i in range(min(3, edge_index.shape[1])):
                src_idx = edge_index[0, i].item()
                dst_idx = edge_index[1, i].item()
                print(f"      {src_idx} -> {dst_idx}")

# 执行查看
inspect_hetero_data(train_data, "训练集 (Train)")
inspect_hetero_data(val_data, "验证集 (Val)")


======================================== 训练集 (Train) 数据详情 ========================================
🔹 包含的节点类型: ['gene/protein', 'drug', 'disease', 'effect/phenotype', 'biological_process', 'molecular_function', 'cellular_component', 'exposure', 'pathway', 'anatomy']
🔹 包含的边类型: [('gene/protein', 'protein_protein', 'gene/protein'), ('drug', 'drug_protein', 'gene/protein'), ('drug', 'contraindication', 'disease'), ('drug', 'indication', 'disease'), ('drug', 'off-label use', 'disease'), ('drug', 'drug_drug', 'drug'), ('gene/protein', 'phenotype_protein', 'effect/phenotype'), ('effect/phenotype', 'phenotype_phenotype', 'effect/phenotype'), ('disease', 'disease_phenotype_negative', 'effect/phenotype'), ('disease', 'disease_phenotype_positive', 'effect/phenotype'), ('effect/phenotype', 'disease_phenotype_positive', 'disease'), ('gene/protein', 'disease_protein', 'disease'), ('disease', 'disease_disease', 'disease'), ('drug', 'drug_effect', 'effect/phenotype'), ('biological_process', 'bioproces